# Data preparation of generation data from ENTSO-E transparency

source: https://transparency.entsoe.eu/

Creates the following parsed datasets

- Hourly generation per technology and country for selected year (generation_"+year+"_hourly_entsoe.csv) 
- Weekly generation per technology and country for selected year (generation_"+year+"_weekly_entsoe.csv) 
- Monthly generation per technology and country for selected year (generation_"+year+"_monthly_entsoe.csv) 
- Yearly generation per technology and country for selected year (generation_"+year+"_annual_entsoe.csv) 

Settings in next window

In [22]:
#download files again (yes/no)?
download = "yes"

#set year for data creation
year = '2024'

In [23]:
import pysftp
import sys
import os
import pandas as pd
import datetime as dt
import plotly.express as px

In [24]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
#enter your password here
password = "3?mJ}V4us?!L}5E"                
#enter email address here
username = "jsavelsberg@ethz.ch"                
port = '22'

In [25]:
cnopts = pysftp.CnOpts()
cnopts.hostkeys = None

c:\Users\jonas\anaconda3\Lib\site-packages\pysftp\__init__.py:61: UserWarning: Failed to load HostKeys from C:\Users\jonas\.ssh\known_hosts.  You will need to explicitly load HostKeys (cnopts.hostkeys.load(filename)) or disableHostKey checking (cnopts.hostkeys = None).
  warnings.warn(wmsg, UserWarning)


In [26]:
dir_out = "../parsed_data/"

In [27]:
#technology definition
dict_agg_tech = {"Other": "Other",
                "Wind Offshore": "WindOffshore",
                "Fossil Brown coal/Lignite": "Lignite",
                "Nuclear": "Nuclear",
                "Fossil Hard coal": "HardCoal",
                "Geothermal": "Other",
                "Fossil Coal-derived gas": "Other",
                "Hydro Pumped Storage": "Pump",
                "Hydro Run-of-river and poundage": "RunOfRiver",
                "Biomass": "Biomass",
                "Fossil Peat": "Other",
                "Fossil Oil shale": "Oil",
                "Fossil Oil": "Oil",
                "Hydro Water Reservoir": "Reservoir",
                "Marine": "Other",
                "Wind Onshore": "WindOnshore",
                "Other renewable": "Other",
                "Solar": "Solar",
                "Waste": "Other",
                "Fossil Gas": "Gas",
                "rooftop_pv" : "Solar",
                "onshore_wind" : "WindOnshore",
                "offshore_wind" : "WindOffshore"}

In [28]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/ENTSOE/') 

In [ ]:
# show list of all available folders (uncomment last line if needed)
# relevant folder was renamed to AggregatedGenerationPerType_16.1.B_C
with pysftp.Connection(host=host, username=username, password=password, cnopts=cnopts) as sftp:
    print("Connection succesfully established.")
    files = sftp.listdir('/TP_export/')   
    print(files)

Connection succesfully established.
['AcceptedAggregatedOffers_17.1.D', 'ActivatedBalancingEnergy_17.1.E', 'ActualCapacitiesAndOutlookOnFrequencyRestorationReserveAndReplacementReserve_SOGL_188.3_188.4_189.2_189.3_r3', 'ActualGenerationOutputPerGenerationUnit_16.1.A_r2.1', 'ActualTotalLoad_6.1.A', 'AggregatedBalancingEnergyBids_12.3.E_r3', 'AggregatedFillingRateOfWaterReservoirsAndHydroStoragePlants_16.1.D', 'AggregatedGenerationPerType_16.1.B_C', 'AmountAndPricesPaidOfBalancingReservesUnderContract_17.1.B_C_r2', 'AmountOfBalancingReservesUnderContract_17.1.B', 'BalancingBorderCapacityLimitations_IFs_4.3_4.4_r2', 'ChangesInActualAvailabilityOfConsumptionUnits_7.1.B', 'ChangesInActualAvailabilityOfOffshoreGridInfrastructureReasons_10.1.C', 'ChangesInActualAvailabilityOfOffshoreGridInfrastructure_10.1.C', 'ChangesToBidAvailability_IFs_mFRR9.9_aFRR9.6_9.8_r3', 'CommercialSchedulesNetPositions_12.1.F_r3', 'CommercialSchedules_12.1.F_r3', 'CostsOfCongestionManagement_13.1.C', 'Countertradin

In [30]:
#load file names from server
path_gen = path+'AggregatedGenerationPerType_16.1.B_C/'
path_gen_local = path_local+'generation/'
with pysftp.Connection(host=host, username=username, password=password, cnopts=cnopts) as sftp:
    print("Connection succesfully established.")
    # show list of files
    files = sftp.listdir(path_gen)
    #download files
    if year != "":
        files = [i for i in files if year in i]
print(files)

Connection succesfully established.
['2024_01_AggregatedGenerationPerType_16.1.B_C.csv', '2024_02_AggregatedGenerationPerType_16.1.B_C.csv', '2024_03_AggregatedGenerationPerType_16.1.B_C.csv', '2024_04_AggregatedGenerationPerType_16.1.B_C.csv', '2024_05_AggregatedGenerationPerType_16.1.B_C.csv', '2024_06_AggregatedGenerationPerType_16.1.B_C.csv', '2024_07_AggregatedGenerationPerType_16.1.B_C.csv', '2024_08_AggregatedGenerationPerType_16.1.B_C.csv', '2024_09_AggregatedGenerationPerType_16.1.B_C.csv', '2024_10_AggregatedGenerationPerType_16.1.B_C.csv', '2024_11_AggregatedGenerationPerType_16.1.B_C.csv', '2024_12_AggregatedGenerationPerType_16.1.B_C.csv']


In [31]:
#download aggregated generation data (AggregatedGenerationPerType)
if download == "yes":
    with pysftp.Connection(host=host, username=username, password=password, cnopts=cnopts) as sftp:
        for file in files:
            sftp.get(path_gen+file,path_gen_local+file)
            print('Successfully downloaded file '+file)

Successfully downloaded file 2024_01_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_02_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_03_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_04_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_05_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_06_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_07_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_08_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_09_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_10_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_11_AggregatedGenerationPerType_16.1.B_C.csv


Exception ignored in: <function Connection.__del__ at 0x000001D550005A80>
Traceback (most recent call last):
  File "c:\Users\jonas\anaconda3\Lib\site-packages\pysftp\__init__.py", line 1013, in __del__
    self.close()
  File "c:\Users\jonas\anaconda3\Lib\site-packages\pysftp\__init__.py", line 784, in close
    if self._sftp_live:
       ^^^^^^^^^^^^^^^
AttributeError: 'Connection' object has no attribute '_sftp_live'
Exception ignored in: <function Connection.__del__ at 0x000001D550005A80>
Traceback (most recent call last):
  File "c:\Users\jonas\anaconda3\Lib\site-packages\pysftp\__init__.py", line 1013, in __del__
    self.close()
  File "c:\Users\jonas\anaconda3\Lib\site-packages\pysftp\__init__.py", line 784, in close
    if self._sftp_live:
       ^^^^^^^^^^^^^^^
AttributeError: 'Connection' object has no attribute '_sftp_live'


Successfully downloaded file 2024_12_AggregatedGenerationPerType_16.1.B_C.csv


In [9]:
#combine files to one data frame
df_gen_in = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_gen_local+file,
                          decimal=".",encoding="UTF-16LE",sep="\t",
                          parse_dates=True, index_col="DateTime")
    df_gen_in = df_gen_in.append(df_temp)
#country values for HR are missing so we change this to area type code
df_gen_in.loc[((df_gen_in.MapCode == 'HR') & (df_gen_in.AreaTypeCode == 'CTY')),'AreaTypeCode'] = 'not_it'
df_gen_in.loc[((df_gen_in.MapCode == 'HR') & (df_gen_in.AreaTypeCode == 'BZN')),'AreaTypeCode'] = 'CTY'
df_gen_in = df_gen_in[df_gen_in.AreaTypeCode == "CTY"].drop(["areacode","AreaTypeCode",'Year','Month','Day'], axis=1).reset_index()
df_gen_in = df_gen_in.sort_values(by=['DateTime'])
df_gen_in["technology"] = df_gen_in.ProductionType.map(dict_agg_tech)
df_gen_in.info()

FileNotFoundError: [Errno 2] No such file or directory: '..\\source_data/ENTSOE/generation/AcceptedAggregatedOffers_17.1.D'

In [11]:
df_temp = pd.read_csv(path_gen_local+files[10],
                      decimal=".",encoding="UTF-16LE",sep="\t",
                      parse_dates=True, index_col="DateTime")
df_temp.head()


,Year,Month,Day,ResolutionCode,areacode,AreaTypeCode,AreaName,MapCode,ProductionType,ActualGenerationOutput,ActualConsumption,UpdateTime
DateTime,,,,,,,,,,,,
2020-08-28 06:00:00,2020,8,28,PT60M,10YFR-RTE------C,CTA,RTE CA,FR,Fossil Hard coal,499.0,NaN,2020-08-28 09:47:14
2020-08-28 06:00:00,2020,8,28,PT60M,10YFR-RTE------C,CTA,RTE CA,FR,Hydro Water Reservoir,701.0,NaN,2020-08-28 09:47:14
2020-08-28 06:00:00,2020,8,28,PT60M,10YFR-RTE------C,CTA,RTE CA,FR,Hydro Pumped Storage,348.0,NaN,2020-08-28 09:47:14
2020-08-28 06:00:00,2020,8,28,PT60M,10YFR-RTE------C,CTA,RTE CA,FR,Fossil Oil,140.0,NaN,2020-08-28 09:47:15
2020-08-28 06:00:00,2020,8,28,PT60M,10YFR-RTE------C,CTA,RTE CA,FR,Solar,743.0,NaN,2020-08-28 09:47:15


In [12]:
df_temp.AreaName.unique()

array(['RTE CA', 'TenneT NL BZ', 'MAVIR CA', 'Hungary', 'Greece',
       'APG BZ', 'IT-Centre-North BZ', 'IT-North BZ', '50Hertz CA',
       'TenneT GER CA', 'DK2 BZ', 'Energinet CA', 'United Kingdom',
       'National Grid BZ', 'Republic of Moldova', 'AST BZ', 'SE4 BZ',
       'SvK CA', 'SE1 BZ', 'SE2 BZ', 'CEPS BZ', 'Elia CA', 'Belgium',
       'Poland', 'Elering BZ', 'Litgrid CA', 'NO3 BZ', 'REE BZ',
       'North Macedonia', 'HOPS BZ', 'swissgrid BZ', 'TenneT NL CA',
       'Netherlands', 'Elia BZ', 'Cyprus TSO CA', 'NO1 BZ', 'REN BZ',
       'Sweden', 'Germany', 'EirGrid CA', 'SEPS CA', 'MAVIR BZ',
       'National Grid CA', 'Romania', 'Estonia', 'France', 'IPTO BZ',
       'IT-Centre-South BZ', 'IT-Rossano BZ', 'Italy CA', 'ELES CA',
       'ELES BZ', 'Denmark', 'EMS CA', 'swissgrid CA', 'MD BZ', 'Finland',
       'Bulgaria', 'ESO BZ', 'MEPSO BZ', 'HOPS CA', 'Amprion CA',
       'PSE SA BZ', 'Cyprus', 'Austria', 'RTE BZ', 'IT-Sicily BZ',
       'Italy', 'Switzerland', 'AST CA', '

In [13]:
df_temp.MapCode.unique()

array(['FR', 'NL', 'HU', 'GR', 'AT', 'IT_CNOR', 'IT_North', 'DE_50HzT',
       'DE_TenneT_GER', 'DK2', 'DK', 'GB', 'MD', 'LV', 'SE4', 'SE', 'SE1',
       'SE2', 'CZ', 'BE', 'PL', 'EE', 'LT', 'NO3', 'ES', 'MK', 'HR', 'CH',
       'CY', 'NO1', 'PT', 'DE', 'IE', 'SK', 'RO', 'IT_CSUD', 'IT_ROSN',
       'IT', 'SI', 'RS', 'FI', 'BG', 'DE_Amprion', 'IT_SICI', 'NO4',
       'NO5', 'NO', 'ME', 'BA', 'DK1', 'DE_LU', 'DE_TransnetBW', 'SE3',
       'NO2', 'NIE', 'IE_SEM', 'IT_SUD', 'IT_SARD'], dtype=object)

In [14]:
#Aggregate technologies:
df_gen = df_gen_in.groupby(["DateTime", "MapCode", "technology"], as_index=False).sum()
df_gen.columns = ["date", "country", "tech", "output", "demand"]
df_gen["net_generation"] = df_gen.output - df_gen.demand
df_gen.info(null_counts=True)

<ipython-input-14-5570b64fc06e>:5: FutureWarning: null_counts is deprecated. Use show_counts instead
  df_gen.info(null_counts=True)


<class 'pandas.core.frame.DataFrame'>
Int64Index: 3653877 entries, 0 to 3653876
Data columns (total 6 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   date            3653877 non-null  datetime64[ns]
 1   country         3653877 non-null  object        
 2   tech            3653877 non-null  object        
 3   output          3653877 non-null  float64       
 4   demand          3653877 non-null  float64       
 5   net_generation  3653877 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(2)
memory usage: 195.1+ MB


In [15]:
df_gen = df_gen.set_index(['date','country','tech'])
df_gen.head()

output  demand  net_generation
date       country tech                                     
2020-01-01 AT      Biomass    208.00     0.0          208.00
                   Gas       1797.60     0.0         1797.60
                   HardCoal   160.00     0.0          160.00
                   Oil          0.00     0.0            0.00
                   Other      122.07     0.0          122.07

In [16]:
#some values are reported quarter hourly so we have to resample to hourly values
df_gen_hourly = df_gen.groupby([pd.Grouper(level='country'),
                                pd.Grouper(level='tech'), 
                                pd.Grouper(level='date', freq='1H')]
                               ).mean()
df_gen_hourly.head()

output  demand  net_generation
country tech    date                                               
AT      Biomass 2020-01-01 00:00:00   208.0     0.0           208.0
                2020-01-01 01:00:00   208.0     0.0           208.0
                2020-01-01 02:00:00   208.0     0.0           208.0
                2020-01-01 03:00:00   207.0     0.0           207.0
                2020-01-01 04:00:00   204.0     0.0           204.0

In [17]:
#We are primarily interested in renewable data. Let's see how complete they are
df_ = df_gen_hourly.groupby(["country", "tech"]).net_generation.count()
df_ = df_.reset_index().pivot_table(index=["country"], columns="tech", values="net_generation")
df_.loc[(slice(None)), ["Solar", "WindOnshore", "WindOffshore", "RunOfRiver","Biomass"]]

tech,Solar,WindOnshore,WindOffshore,RunOfRiver,Biomass
country,,,,,
AT,8784.0,8784.0,NaN,8784.0,8784.0
BA,NaN,8757.0,NaN,NaN,NaN
BE,8784.0,8784.0,8784.0,8784.0,8784.0
BG,8784.0,8784.0,NaN,8712.0,8736.0
CH,8784.0,8784.0,NaN,8784.0,NaN
CY,NaN,7698.0,NaN,NaN,NaN
CZ,8784.0,8784.0,NaN,8784.0,8784.0
DE,8784.0,8784.0,8784.0,8784.0,8784.0
DK,8782.0,8783.0,8782.0,NaN,8782.0


In [18]:
#export annual and monthly data for checking of the data quality
df_gen_a = df_gen_hourly.groupby(["country", "tech"]).sum()/1000000
df_gen_a.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 273 entries, ('AT', 'Biomass') to ('SK', 'WindOnshore')
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   output          273 non-null    float64
 1   demand          273 non-null    float64
 2   net_generation  273 non-null    float64
dtypes: float64(3)
memory usage: 7.4+ KB


In [19]:
df_gen_m = df_gen_hourly.groupby([pd.Grouper(freq='M', level='date'), "country", "tech"]).sum()
df_gen_m.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 3255 entries, (Timestamp('2020-01-31 00:00:00', freq='M'), 'AT', 'Biomass') to (Timestamp('2020-12-31 00:00:00', freq='M'), 'SK', 'WindOnshore')
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   output          3255 non-null   float64
 1   demand          3255 non-null   float64
 2   net_generation  3255 non-null   float64
dtypes: float64(3)
memory usage: 86.4+ KB


In [20]:
#we also export weekly values
df_gen_w = df_gen_hourly.groupby([pd.Grouper(freq='W', level='date'), "country", "tech"]).sum()
df_gen_w.head()

output  demand  net_generation
date       country tech                                      
2020-01-05 AT      Biomass    24696.0     0.0         24696.0
                   Gas       224207.3     0.0        224207.3
                   HardCoal   19324.9     0.0         19324.9
                   Oil            0.0     0.0             0.0
                   Other      14648.4     0.0         14648.4

In [21]:
df_gen_hourly.to_csv(dir_out + "generation_"+year+"_hourly_entsoe.csv", index=True)
df_gen_w.to_csv(dir_out + "generation_"+year+"_weekly_entsoe.csv", index=True)
df_gen_a.to_csv(dir_out + "generation_"+year+"_annual_entsoe.csv", index=True)
df_gen_m.to_csv(dir_out + "generation_"+year+"_monthly_entsoe.csv", index=True)

In [22]:
df_gen.reset_index().country.unique()

array(['AT', 'BA', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES',
       'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LV', 'ME',
       'MK', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'MD'],
      dtype=object)